# Deployment

**Tarefas:** T4.x · **Dono:** _(a definir)_

Bancada de trabalho: a lógica reutilizável fica em `src/`; aqui corre-se, explora-se e
guarda-se o que o `Projeto_Final.ipynb` vai ler (`reports/experiments.csv`, `reports/figures/`).
Regras da modelação: [docs/modelacao_base.md](../docs/modelacao_base.md).

Refit no treino completo e gravação em `models/v1/` (não versionado): `model.skops`, `config.json` (thresholds a 10% e 20% e quantis das OOF) e `README.md` (model card). Teste da app (`src/app.py`).

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de src/ (data/, reports/, models/) são relativos à raiz

import pandas as pd

from src.app import describe, predict_risk
from src.models.baselines import candidate_models
from src.models.train import fit_final, load_model, save_model
from src.pipeline import load_model_data

data = load_model_data()

## Refit e gravação

In [2]:
best = candidate_models()["hist_gradient_boosting"]
oof = pd.read_csv(ROOT / "reports" / "oof_v1.csv")["oof_score"].to_numpy()  # gravado no 04
path = save_model(fit_final(best, data), data, oof, model_name="hist_gradient_boosting")
sorted(p.name for p in path.iterdir())

['README.md', 'config.json', 'model.skops']

## Teste da app

In [3]:
bundle = load_model()
for case in [dict(number_inpatient=0, number_emergency=0, number_outpatient=0),
             dict(number_inpatient=3, number_emergency=1, number_outpatient=0)]:
    print(case, "->", describe(predict_risk(bundle, **case)), sep="\n", end="\n\n")

{'number_inpatient': 0, 'number_emergency': 0, 'number_outpatient': 0}
->
Esta alta está nos 54% com maior risco de readmissão em menos de 30 dias.
Com capacidade de 10%: sem follow-up.
Com capacidade de 20%: sem follow-up.

{'number_inpatient': 3, 'number_emergency': 1, 'number_outpatient': 0}
->
Esta alta está nos 8% com maior risco de readmissão em menos de 30 dias.
Com capacidade de 10%: propor follow-up.
Com capacidade de 20%: propor follow-up.



In [4]:
# from src.app import build_app; build_app(bundle).launch()  # ou: python -m src.app